<a href="https://colab.research.google.com/github/Edubel77/enrutaia/blob/main/notebooks/D6_lab_2_1_spacy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 22.5 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# Lab 2.1 - Paso 1: el pipeline de spaCy en español
import spacy
import pandas as pd

nlp = spacy.load("es_core_news_sm")
print("spaCy", spacy.__version__, "| componentes del pipeline:", nlp.pipe_names)

frases = [
    "Pon las alarmas a las siete de la mañana",
    "¿Va a llover mañana en Bogotá?",
    "Reproduce la última canción de Shakira",
    "Envíale un correo a Daniel diciendo que llego tarde",
]

for frase in frases:
    doc = nlp(frase)
    print(f"\n» {frase}")
    tabla = pd.DataFrame([(t.text, t.lemma_, t.pos_, t.is_stop) for t in doc],
                         columns=["token", "lema", "POS", "¿stopword?"])
    print(tabla.to_string(index=False))
    print("Entidades:", [(e.text, e.label_) for e in doc.ents] or "ninguna")

spaCy 3.8.18 | componentes del pipeline: ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

» Pon las alarmas a las siete de la mañana
  token   lema  POS  ¿stopword?
    Pon    pon VERB       False
    las     el  DET        True
alarmas alarma NOUN       False
      a      a  ADP        True
    las     el  DET        True
  siete  siete  NUM        True
     de     de  ADP        True
     la     el  DET        True
 mañana mañana NOUN       False
Entidades: ninguna

» ¿Va a llover mañana en Bogotá?
 token   lema   POS  ¿stopword?
     ¿      ¿ PUNCT       False
    Va     ir   AUX        True
     a      a   ADP        True
llover llover  VERB       False
mañana mañana   ADV       False
    en     en   ADP        True
Bogotá Bogotá PROPN       False
     ?      ? PUNCT       False
Entidades: [('¿Va', 'ORG'), ('Bogotá', 'LOC')]

» Reproduce la última canción de Shakira
    token       lema   POS  ¿stopword?
Reproduce reproducir  VERB       False
       l

In [3]:
# Lab 2.1 - Paso 2: perfilado lingüístico de train
import hashlib, io, urllib.request
from collections import Counter

BASE = "https://raw.githubusercontent.com/Edubel77/enrutaia/main/data/limpio/"
contenido = urllib.request.urlopen(BASE + "train.csv").read()
assert hashlib.md5(contenido).hexdigest() == "cc0ff7796f8e1d1df56862a9f708d521", "¡train.csv cambió!"
train = pd.read_csv(io.BytesIO(contenido))

# ¿Cómo vienen escritos los textos?
print("Textos que empiezan con mayúscula:", f"{train['text'].str[0].str.isupper().mean():.1%}")
print("Ejemplos:", train["text"].sample(5, random_state=1).tolist())

# Procesar todo train (sin el parser sintáctico, que no necesitamos: es más rápido)
docs = list(nlp.pipe(train["text"], batch_size=256, disable=["parser"]))

# 1) Categorías gramaticales
pos = Counter(t.pos_ for d in docs for t in d)
total = sum(pos.values())
print("\nPOS más frecuentes:", [(p, f"{n / total:.1%}") for p, n in pos.most_common(6)])
print(f"Proporción de stopwords: {sum(t.is_stop for d in docs for t in d) / total:.1%}")

# 2) Lemas más característicos por dominio (sin stopwords ni puntuación)
train["lemas"] = [[t.lemma_.lower() for t in d if t.is_alpha and not t.is_stop] for d in docs]
for dom in ["alarm", "weather", "general", "qa"]:
    c = Counter(l for ls in train.loc[train["dominio"] == dom, "lemas"] for l in ls)
    print(f"\n{dom:<8}:", [l for l, _ in c.most_common(8)])

# 3) Entidades
ents = Counter(e.label_ for d in docs for e in d.ents)
print("\nEntidades por tipo:", dict(ents))
print("Textos con al menos una entidad:", f"{sum(bool(d.ents) for d in docs) / len(docs):.1%}")
print("Ejemplos de entidades:", [(e.text, e.label_) for d in docs[:300] for e in d.ents][:12])

Textos que empiezan con mayúscula: 0.0%
Ejemplos: ['jugar los veinticinco mejores éxitos', 'por favor elimine los plátanos de mi lista de compras', 'estoy bien', 'encontrar zapatería cerca de mi casa', 'hay algún eventos sucediendo en la ciudad']

POS más frecuentes: [('NOUN', '26.5%'), ('ADP', '17.2%'), ('DET', '16.9%'), ('VERB', '12.8%'), ('ADJ', '6.7%'), ('PRON', '4.4%')]
Proporción de stopwords: 53.0%

alarm   : ['alarma', 'mañana', 'favor', 'hora', 'pon', 'puesto', 'borrar', 'necesitar']

weather : ['tiempo', 'semana', 'necesitar', 'mañana', 'temperatura', 'llover', 'pronóstico', 'deber']

general : ['chiste', 'poder', 'broma', 'oir', 'tener', 'favor', 'pasar', 'querer']

qa      : ['acción', 'dólar', 'precio', 'cambio', 'tipo', 'año', 'definición', 'estadounidense']

Entidades por tipo: {'ORG': 453, 'PER': 1919, 'MISC': 856, 'LOC': 1266}
Textos con al menos una entidad: 34.8%
Ejemplos de entidades: [('pon', 'ORG'), ('olly', 'PER'), ('olly', 'PER'), ('pon', 'ORG'), ('olly', 'PER')

In [4]:
!mkdir -p src tests

In [6]:
%%writefile src/text_clean.py
"""Preprocesamiento lingüístico de EnrutaIA con spaCy (Día 6)."""


def lematizar(textos, nlp, quitar_stopwords=False):
    """Devuelve cada texto como sus lemas en minúscula, separados por espacios.

    - Elimina la puntuación.
    - Si quitar_stopwords=True, elimina también las palabras vacías ("de", "la"...).
    - Devuelve una lista del mismo largo y en el mismo orden que `textos`.
    """
    salida = []
    for doc in nlp.pipe(list(textos), batch_size=256, disable=["parser", "ner"]):
        lemas = [t.lemma_.lower() for t in doc
                 if not t.is_punct and not t.is_space
                 and not (quitar_stopwords and t.is_stop)]
        salida.append(" ".join(lemas))
    return salida

Overwriting src/text_clean.py


In [11]:
%%writefile tests/test_text_clean.py
import pytest
import spacy
from src.text_clean import lematizar


@pytest.fixture(scope="module")
def nlp():
    """Carga el modelo de spaCy UNA sola vez para todas las pruebas del archivo."""
    return spacy.load("es_core_news_sm")


def test_lleva_plurales_a_singular(nlp):
    assert "alarma" in lematizar(["pon las alarmas"], nlp)[0].split()


def test_devuelve_minusculas(nlp):
    resultado = lematizar(["Bogotá"], nlp)[0]
    assert resultado == resultado.lower()


def test_conserva_largo_y_orden(nlp):
    textos = ["pon música", "qué hora es", "enciende la luz"]
    salida = lematizar(textos, nlp)
    assert len(salida) == 3
    assert "luz" in salida[2]


def test_elimina_puntuacion(nlp):
    assert "?" not in lematizar(["¿qué hora es?"], nlp)[0]


def test_texto_vacio(nlp):
    assert lematizar([""], nlp) == [""]

def test_lematizar_quitar_stopwords(nlp):
    textos = ["pon la alarma de la mañana"]
    resultado = lematizar(textos, nlp, quitar_stopwords=True)
    palabras = resultado[0].split()

    assert "la" not in palabras
    assert "de" not in palabras
    assert "alarma" in palabras

Overwriting tests/test_text_clean.py


In [12]:
!pip install -q pytest
!python -m pytest tests/test_text_clean.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 6 items                                                              

tests/test_text_clean.py::test_lleva_plurales_a_singular PASSED          [ 16%]
tests/test_text_clean.py::test_devuelve_minusculas PASSED                [ 33%]
tests/test_text_clean.py::test_conserva_largo_y_orden PASSED             [ 50%]
tests/test_text_clean.py::test_elimina_puntuacion PASSED                 [ 66%]
tests/test_text_clean.py::test_texto_vacio PASSED                        [ 83%]
tests/test_text_clean.py::test_lematizar_quitar_stopwords PASSED         [100%]

============================== 6 passed in 3.80s ===============================


In [14]:
!mkdir -p results

In [15]:
# Lab 2.1 - Paso 4: ¿la lematización mejora E0?
import json, time
from sklearn.pipeline import make_pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from src.text_clean import lematizar

contenido = urllib.request.urlopen(BASE + "validation.csv").read()
assert hashlib.md5(contenido).hexdigest() == "ba68845308293612e717aa7c1a55887c", "¡validation.csv cambió!"
val = pd.read_csv(io.BytesIO(contenido))

def crear_e0():
    """Exactamente la misma configuración de E0 (Día 4)."""
    return make_pipeline(
        FeatureUnion([
            ("palabras",   TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
            ("caracteres", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
        ]),
        LogisticRegression(max_iter=2000),
    )

variantes = {
    "E0 (texto original)": (train["text"].tolist(), val["text"].tolist()),
    "E0-lemas":            (lematizar(train["text"], nlp), lematizar(val["text"], nlp)),
    "E0-lemas-sin-stop":   (lematizar(train["text"], nlp, quitar_stopwords=True),
                            lematizar(val["text"], nlp, quitar_stopwords=True)),
}

print("Ejemplo de cada versión:")
for nombre, (tr, _) in variantes.items():
    print(f"  {nombre:<20} -> {tr[1]}")

resultados = {}
print()
for nombre, (tr, va) in variantes.items():
    modelo = crear_e0()
    modelo.fit(tr, train["dominio"])
    f1 = f1_score(val["dominio"], modelo.predict(va), average="macro")
    resultados[nombre] = round(f1, 4)
    print(f"{nombre:<20} | macro-F1 validación: {f1:.4f} | diferencia vs E0: {f1 - resultados['E0 (texto original)']:+.4f}")

with open("results/E0_variantes_spacy.json", "w") as f:
    json.dump({"experimento": "E0 con preprocesamiento spaCy (es_core_news_sm)",
               "split_evaluado": "validation", "macro_f1": resultados,
               "spacy": spacy.__version__}, f, indent=2, ensure_ascii=False)

Ejemplo de cada versión:
  E0 (texto original)  -> pon una alarma en dos horas desde ahora
  E0-lemas             -> pon uno alarma en dos hora desde ahora
  E0-lemas-sin-stop    -> pon alarma hora

E0 (texto original)  | macro-F1 validación: 0.8725 | diferencia vs E0: +0.0000
E0-lemas             | macro-F1 validación: 0.8764 | diferencia vs E0: +0.0039
E0-lemas-sin-stop    | macro-F1 validación: 0.8451 | diferencia vs E0: -0.0274


In [16]:
import spacy
print(f"spacy=={spacy.__version__}")
!pip show es_core_news_sm | grep Version

spacy==3.8.18
Version: 3.8.0
